# 🛒 Retail NLP Pipeline: Customer Feedback Sentiment Analysis

**Project Objective:**
Automate the processing and categorization of daily grocery delivery feedback. Instead of manually reading through hundreds of support tickets, this pipeline uses Natural Language Processing (NLP) to instantly flag operational issues—like damaged fresh produce or missing items—and route them to the correct teams for immediate action.

**Core Workflow:**
* **1. Text Sanitization:** Standardize raw user inputs by converting to lowercase, stripping punctuation, and removing noisy "stop words" to create machine-readable data.
* **2. Sentiment Scoring:** Deploy VADER (Valence Aware Dictionary and sEntiment Reasoner) to mathematically grade the emotional tone of each review on a scale from highly negative (-1.0) to highly positive (+1.0).
* **3. Action Routing:** Map sentiment scores to automated business rules, instantly tagging tickets as **Escalate to Support**, **Operations Alert**, or **Positive Testimonial**.

In [3]:
import pandas as pd

# 1. Generate a mock dataset of unstructured daily grocery delivery reviews
raw_feedback = {
    "ticket_id": [1001, 1002, 1003, 1004, 1005, 1006],
    "review_text": [
        "The fresh spinach was completely wilted and unusable. Very disappointed.",
        "Delivery was right on time at 7 AM! Perfect start to the day.",
        "Missing two packets of whole milk from my morning order.",
        "Excellent quality on the imported apples, will definitely buy again.",
        "App keeps freezing when I try to update my cart for tomorrow.",
        "Delivery executive left the bags in the rain. Bread is ruined!!!"
    ]
}

# 2. Load into a Pandas DataFrame
df = pd.DataFrame(raw_feedback)

# 3. Display the raw text data
display(df)

,ticket_id,review_text
0,1001,The fresh spinach was completely wilted and un...
1,1002,Delivery was right on time at 7 AM! Perfect st...
2,1003,Missing two packets of whole milk from my morn...
3,1004,"Excellent quality on the imported apples, will..."
4,1005,App keeps freezing when I try to update my car...
5,1006,Delivery executive left the bags in the rain. ...


## Text Cleaning Pipeline
This step converts messy customer reviews into machine-readable data. It automatically:

Converts all text to lowercase to ensure uniformity.

Removes standard English stop words (like "and", "the", and "is") which do not carry significant meaning and are usually removed.

In [4]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
import string

# Download the required NLTK dictionary files (only needs to run once)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

def clean_review_text(text):
    # 1. Convert everything to lowercase
    text = text.lower()

    # 2. Remove punctuation (like !!! or ???)
    text = "".join([char for char in text if char not in string.punctuation])

    # 3. Remove filler stop words
    stop_words = set(stopwords.words('english'))
    words = text.split()
    cleaned_words = [word for word in words if word not in stop_words]

    # 4. Rejoin the clean words into a single string
    return " ".join(cleaned_words)

# Apply the idiot-proof cleaning function to our raw feedback
df['cleaned_text'] = df['review_text'].apply(clean_review_text)

# Display the before-and-after comparison
display(df[['review_text', 'cleaned_text']])

,review_text,cleaned_text
0,The fresh spinach was completely wilted and un...,fresh spinach completely wilted unusable disap...
1,Delivery was right on time at 7 AM! Perfect st...,delivery right time 7 perfect start day
2,Missing two packets of whole milk from my morn...,missing two packets whole milk morning order
3,"Excellent quality on the imported apples, will...",excellent quality imported apples definitely buy
4,App keeps freezing when I try to update my car...,app keeps freezing try update cart tomorrow
5,Delivery executive left the bags in the rain. ...,delivery executive left bags rain bread ruined


## Step 2: Sentiment Scoring & Automated Routing
This step uses VADER, a pre-trained lexicon rule-based sentiment analyzer, to assign a mathematical emotion score to each sanitized review. Based on that compound score (-1.0 to +1.0), the pipeline automatically triggers a business action—instantly routing severe issues without manual reading.

In [5]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# 1. Download the VADER lexicon (only needs to run once)
nltk.download('vader_lexicon', quiet=True)

# 2. Initialize the sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# 3. Define the scoring and routing logic
def score_and_route(text):
    # Calculate the compound sentiment score
    score = analyzer.polarity_scores(text)['compound']

    # Assign automated actions based on the score thresholds
    if score <= -0.5:
        return pd.Series([score, "🚨 Escalate to Support"])
    elif score < 0:
        return pd.Series([score, "⚠️ Operations Alert"])
    elif score > 0.4:
        return pd.Series([score, "🌟 Positive Testimonial"])
    else:
        return pd.Series([score, "✅ Standard Review"])

# 4. Apply the function to generate our final metrics
df[['sentiment_score', 'action_flag']] = df['cleaned_text'].apply(score_and_route)

# 5. Display the final actionable dataset
display(df[['ticket_id', 'review_text', 'sentiment_score', 'action_flag']])

,ticket_id,review_text,sentiment_score,action_flag
0,1001,The fresh spinach was completely wilted and un...,-0.2648,⚠️ Operations Alert
1,1002,Delivery was right on time at 7 AM! Perfect st...,0.5719,🌟 Positive Testimonial
2,1003,Missing two packets of whole milk from my morn...,-0.2960,⚠️ Operations Alert
3,1004,"Excellent quality on the imported apples, will...",0.7506,🌟 Positive Testimonial
4,1005,App keeps freezing when I try to update my car...,-0.1027,⚠️ Operations Alert
5,1006,Delivery executive left the bags in the rain. ...,-0.4767,⚠️ Operations Alert


In [6]:
# 1. Download and load spaCy's English language model
!python -m spacy download en_core_web_sm -q
import spacy
nlp = spacy.load("en_core_web_sm")

# 2. Define an NLP rule to extract physical items
def extract_items(text):
    # Process the text through the neural network
    doc = nlp(text.lower())

    # Extract nouns, filtering out generic conversational words
    ignore_words = {'day', 'time', 'start', 'order', 'app', 'cart', 'tomorrow', 'executive', 'rain', 'quality'}
    items = [token.text for token in doc if token.pos_ == "NOUN" and token.text not in ignore_words]

    return ", ".join(items) if items else "System/General"

# 3. Apply the extraction to our dataset
df['flagged_items'] = df['review_text'].apply(extract_items)

# 4. Display the fully autonomous routing table
display(df[['ticket_id', 'action_flag', 'flagged_items']])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 86.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


,ticket_id,action_flag,flagged_items
0,1001,⚠️ Operations Alert,spinach
1,1002,🌟 Positive Testimonial,"delivery, am"
2,1003,⚠️ Operations Alert,"packets, milk, morning"
3,1004,🌟 Positive Testimonial,apples
4,1005,⚠️ Operations Alert,System/General
5,1006,⚠️ Operations Alert,"delivery, bags, bread"


Step 3: Supervised Department Classification (TF-IDF + Naive Bayes)
Converts raw text into numerical features using Term Frequency-Inverse Document Frequency (TF-IDF) and trains a Multinomial Naive Bayes classifier to automatically route tickets to the responsible retail department (Fruits & Vegetables, Dairy, Bakery, Tech & Logistics).

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
import joblib

# 1. Labeled reference dataset for supervised training
training_data = {
    "text": [
        # Fruits & Vegetables
        "fresh spinach wilted spoiled lettuce rotten tomatoes rotten potatoes",
        "crisp sweet apples bananas fresh coriander quality oranges",
        "mushy cucumbers yellow bitter greens stale onions",

        # Dairy
        "spoiled cow milk curdled butter packet broken paneer",
        "missing packets of whole milk toned milk cheese slices yogurt",
        "sour cream fresh buttermilk ghee bottle leaking",

        # Bakery
        "bread loaf crushed moldy brown bread pav burger buns",
        "stale garlic bread dry sliced bread croissants damaged",
        "hard dinner rolls sandwich bread torn packaging",

        # Tech & Logistics
        "delivery executive late rain wet bag delayed morning slot",
        "app keeps crashing payment failed wallet deduction error",
        "cart not updating checkout error server timeout order failed"
    ],
    "department": [
        "Fruits & Vegetables", "Fruits & Vegetables", "Fruits & Vegetables",
        "Dairy", "Dairy", "Dairy",
        "Bakery", "Bakery", "Bakery",
        "Tech & Logistics", "Tech & Logistics", "Tech & Logistics"
    ]
}

# 2. Build and train an end-to-end NLP classification pipeline
classifier_pipeline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),
    MultinomialNB(alpha=0.1)
)

classifier_pipeline.fit(training_data["text"], training_data["department"])

# 3. Predict the target department for our live feedback tickets
df['predicted_department'] = classifier_pipeline.predict(df['cleaned_text'])

# 4. Display the complete triage table
display(df[['ticket_id', 'review_text', 'sentiment_score', 'action_flag', 'predicted_department']])

# 5. Persist the trained model pipeline for API deployment
joblib.dump(classifier_pipeline, "department_classifier.joblib")
print("Model pipeline saved successfully as department_classifier.joblib")

,ticket_id,review_text,sentiment_score,action_flag,predicted_department
0,1001,The fresh spinach was completely wilted and un...,-0.2648,⚠️ Operations Alert,Fruits & Vegetables
1,1002,Delivery was right on time at 7 AM! Perfect st...,0.5719,🌟 Positive Testimonial,Tech & Logistics
2,1003,Missing two packets of whole milk from my morn...,-0.2960,⚠️ Operations Alert,Dairy
3,1004,"Excellent quality on the imported apples, will...",0.7506,🌟 Positive Testimonial,Fruits & Vegetables
4,1005,App keeps freezing when I try to update my car...,-0.1027,⚠️ Operations Alert,Tech & Logistics
5,1006,Delivery executive left the bags in the rain. ...,-0.4767,⚠️ Operations Alert,Tech & Logistics


Model pipeline saved successfully as department_classifier.joblib
